In [9]:
from web3 import Web3
from solcx import compile_source, install_solc
import pandas as pd
import numpy as np
import time

# Connect to Ganache
ganache_url = "HTTP://127.0.0.1:7545"
web3 = Web3(Web3.HTTPProvider(ganache_url))

# Check connection
print(web3.is_connected())

# Set account
web3.eth.default_account = web3.eth.accounts[0]

# Install Solidity compiler
install_solc('0.8.0')

# Solidity Smart Contract
contract_source_code = '''

pragma solidity ^0.8.0;

contract IoTData {

    struct Record {
        uint timestamp;
        string device_id;
        string data_type;
        string data_value;
    }

    Record[] public records;

    function addRecord(
        uint _timestamp,
        string memory _device_id,
        string memory _data_type,
        string memory _data_value
    ) public {

        records.push(
            Record(
                _timestamp,
                _device_id,
                _data_type,
                _data_value
            )
        );
    }

    function getTotalRecords() public view returns(uint) {
        return records.length;
    }

    function getRecord(uint index)
        public
        view
        returns(
            uint,
            string memory,
            string memory,
            string memory
        )
    {
        Record memory r = records[index];

        return (
            r.timestamp,
            r.device_id,
            r.data_type,
            r.data_value
        );
    }
}
'''

# Compile contract
compiled_sol = compile_source(
    contract_source_code,
    solc_version='0.8.0'
)

contract_id, contract_interface = compiled_sol.popitem()

# Get bytecode and ABI
bytecode = contract_interface['bin']
abi = contract_interface['abi']

# Deploy contract
IoTContract = web3.eth.contract(
    abi=abi,
    bytecode=bytecode
)

tx_hash = IoTContract.constructor().transact()

tx_receipt = web3.eth.wait_for_transaction_receipt(tx_hash)

# Contract instance
contract = web3.eth.contract(
    address=tx_receipt.contractAddress,
    abi=abi
)

print("Contract deployed successfully!")

# Sample IoT Data
iot_data = [
    (int(time.time()), "PAT001", "Heart Rate", "75 BPM"),
    (int(time.time()), "PAT002", "Oxygen Level", "98%"),
    (int(time.time()), "PAT003", "Temperature", "36.5°C")
]

# Store records in blockchain
for record in iot_data:

    contract.functions.addRecord(
        record[0],
        record[1],
        record[2],
        record[3]
    ).transact()

print("Data stored successfully!")

# Retrieve total records
total_records = contract.functions.getTotalRecords().call()

print(f"Total IoT records stored: {total_records}")

# Retrieve all records
data = []

for i in range(total_records):

    record = contract.functions.getRecord(i).call()

    data.append({
        "timestamp": record[0],
        "device_id": record[1],
        "data_type": record[2],
        "data_value": record[3]
    })

# Convert to DataFrame
df = pd.DataFrame(data)

# Convert timestamp
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    unit="s"
)

# Extract numeric values
df["numeric_value"] = (
    df["data_value"]
    .str.extract(r'(\d+\.?\d*)')[0]
    .astype(float)
)

# Handle missing values
df.fillna(0, inplace=True)

# Display output
print(df.head())

# Save CSV
df.to_csv(
    "cleaned_iot_data.csv",
    index=False
)

print("✅ Cleaned IoT data saved successfully!")

True
Contract deployed successfully!
Data stored successfully!
Total IoT records stored: 3
            timestamp device_id     data_type data_value  numeric_value
0 2026-06-07 14:43:18    PAT001    Heart Rate     75 BPM           75.0
1 2026-06-07 14:43:18    PAT002  Oxygen Level        98%           98.0
2 2026-06-07 14:43:18    PAT003   Temperature     36.5°C           36.5
✅ Cleaned IoT data saved successfully!
